# Data Consumption — Delta Lake discovery

Discover and inspect Delta tables synced from each pipeline zone’s `metadata/observations.parquet`:

| Zone | Bucket (default) | Delta path |
|------|------------------|------------|
| landing | `landing-zone` | `metadata/observations_delta/` |
| trusted | `trusted-zone` | `metadata/observations_delta/` |
| exploitation | `exploitation-zone` | `metadata/observations_delta/` |

Requires MinIO (`docker compose up -d minio`) and at least one zone processing run that completed Delta sync.

## Paths and environment

In [ ]:
from pathlib import Path
import os
import sys

_here = Path.cwd().resolve()
_root = None
for p in [_here, *_here.parents]:
    if (p / "shared" / "minio_helpers.py").is_file():
        _root = p
        break
if _root is None:
    raise RuntimeError("Repo root not found — open the notebook from the BDM-Cymatics tree")

PROJECT_ROOT = str(_root)
DATA_CONSUMPTION_DIR = _root / "data_consumption"

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
if str(DATA_CONSUMPTION_DIR) not in sys.path:
    sys.path.insert(1, str(DATA_CONSUMPTION_DIR))

from dotenv import load_dotenv
load_dotenv(_root / ".env", override=False)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("MINIO_ENDPOINT =", os.environ.get("MINIO_ENDPOINT", "localhost:9000"))

## Zone table registry

In [ ]:
from shared.minio_helpers import LANDING_ZONE_BUCKET, MINIO_ENDPOINT, create_minio_client
from shared.sync_delta import OBSERVATIONS_DELTA_PATH, s3_storage_options

STORAGE_OPTIONS = s3_storage_options()

ZONE_TABLES = {
    "landing": {
        "bucket": LANDING_ZONE_BUCKET,
        "delta_path": OBSERVATIONS_DELTA_PATH,
    },
    "trusted": {
        "bucket": os.environ.get("TRUSTED_ZONE_BUCKET", "trusted-zone"),
        "delta_path": OBSERVATIONS_DELTA_PATH,
    },
    "exploitation": {
        "bucket": os.environ.get("EXPLOITATION_ZONE_BUCKET", "exploitation-zone"),
        "delta_path": OBSERVATIONS_DELTA_PATH,
    },
}

for zone, cfg in ZONE_TABLES.items():
    uri = f"s3a://{cfg['bucket']}/{cfg['delta_path']}"
    print(f"  {zone:14} {uri}")

## Discover tables (row counts + schema)

In [ ]:
from deltalake import DeltaTable


def delta_table_uri(bucket: str, delta_path: str = OBSERVATIONS_DELTA_PATH) -> str:
    return f"s3a://{bucket}/{delta_path}"


def minio_has_delta_log(client, bucket: str, delta_path: str = OBSERVATIONS_DELTA_PATH) -> bool:
    """Quick check: Delta transaction log present on MinIO."""
    prefix = delta_path.rstrip("/") + "/_delta_log/"
    try:
        for _ in client.list_objects(bucket, prefix=prefix, recursive=True):
            return True
    except Exception:
        return False
    return False


def inspect_delta_table(zone: str, bucket: str, delta_path: str = OBSERVATIONS_DELTA_PATH):
    """Return summary dict or error message for one zone Delta table."""
    uri = delta_table_uri(bucket, delta_path)
    client = create_minio_client()
    if not client.bucket_exists(bucket):
        return {"zone": zone, "uri": uri, "status": "missing_bucket"}
    if not minio_has_delta_log(client, bucket, delta_path):
        return {"zone": zone, "uri": uri, "status": "no_delta_table"}
    try:
        dt = DeltaTable(uri, storage_options=STORAGE_OPTIONS)
        version = dt.version()
        pdf = dt.to_pandas()
        return {
            "zone": zone,
            "uri": uri,
            "status": "ok",
            "version": version,
            "num_rows": len(pdf),
            "num_columns": len(pdf.columns),
            "columns": list(pdf.columns),
            "dataframe": pdf,
        }
    except Exception as e:
        return {"zone": zone, "uri": uri, "status": "error", "error": str(e)}


summaries = []
loaded = {}

print(f"MinIO: {MINIO_ENDPOINT}\n")
print(f"{'Zone':<14} {'Status':<16} {'Rows':>8} {'Cols':>6}  URI")
print("-" * 90)

for zone, cfg in ZONE_TABLES.items():
    info = inspect_delta_table(zone, cfg["bucket"], cfg["delta_path"])
    summaries.append(info)
    status = info["status"]
    if status == "ok":
        loaded[zone] = info["dataframe"]
        rows, cols = info["num_rows"], info["num_columns"]
        print(f"{zone:<14} {'ready':<16} {rows:>8} {cols:>6}  {info['uri']}")
    elif status == "no_delta_table":
        print(f"{zone:<14} {'no delta table':<16} {'—':>8} {'—':>6}  {info['uri']}")
    elif status == "missing_bucket":
        print(f"{zone:<14} {'bucket missing':<16} {'—':>8} {'—':>6}  {info['uri']}")
    else:
        print(f"{zone:<14} {'error':<16} {'—':>8} {'—':>6}  {info['uri']}")
        print(f"             {info.get('error', '')}")

## Exploitation zone — sample rows

Full feature-enriched metadata (Spark + Python features). Adjust `PREVIEW_ROWS` or filter by `uuid` below.

In [ ]:
import pandas as pd

PREVIEW_ROWS = 10
UUID_FILTER = ""  # optional: paste a uuid to filter, e.g. "feca5971-..."

if "exploitation" not in loaded:
    print(
        "Exploitation Delta table not available. "
        "Run exploitation_zone_processing.py first (includes Parquet → Delta sync)."
    )
else:
    exp = loaded["exploitation"]
    if UUID_FILTER.strip():
        mask = exp["uuid"].astype(str).str.contains(UUID_FILTER.strip(), case=False, na=False)
        exp = exp.loc[mask]
        print(f"Filtered to {len(exp)} row(s) matching uuid fragment {UUID_FILTER!r}")
    print(f"Total rows: {len(exp)}  |  columns: {len(exp.columns)}")
    display(exp.head(PREVIEW_ROWS))

## Column overview (exploitation)

In [ ]:
if "exploitation" in loaded:
    exp = loaded["exploitation"]
    feature_cols = [
        c
        for c in exp.columns
        if c
        in {
            "spectral_centroid_hz",
            "spectral_bandwidth_hz",
            "spectral_rolloff_hz",
            "spectral_flatness",
            "signal_energy",
            "spectral_entropy",
            "zero_crossing_rate",
            "loudness",
            "MFCCs",
            "harmonic_energy_ratio",
        }
    ]
    meta = pd.DataFrame(
        {
            "column": exp.columns,
            "dtype": [str(exp[c].dtype) for c in exp.columns],
            "non_null": [int(exp[c].notna().sum()) for c in exp.columns],
        }
    )
    display(meta)
    if feature_cols:
        print("\nDerived feature columns:", ", ".join(feature_cols))
        display(exp[feature_cols].describe().T)

## Compare zones (uuid overlap)

Shows how many UUIDs appear in each zone’s Delta table (pipeline should grow landing → trusted → exploitation).

In [ ]:
if not loaded:
    print("No Delta tables loaded — run zone processing + sync first.")
else:
    uuid_sets = {
        zone: set(df["uuid"].astype(str).str.strip())
        for zone, df in loaded.items()
        if "uuid" in df.columns
    }
    for zone, uuids in uuid_sets.items():
        print(f"  {zone}: {len(uuids)} uuid(s)")
    if len(uuid_sets) >= 2:
        zones = list(uuid_sets.keys())
        for i, a in enumerate(zones):
            for b in zones[i + 1 :]:
                inter = uuid_sets[a] & uuid_sets[b]
                print(f"  {a} ∩ {b}: {len(inter)} shared uuid(s)")

## Optional — other zones preview

In [ ]:
ZONE_PREVIEW = "landing"  # landing | trusted | exploitation
N = 5

if ZONE_PREVIEW not in loaded:
    print(f"No data for zone {ZONE_PREVIEW!r}.")
else:
    df = loaded[ZONE_PREVIEW]
    cols = [c for c in ["uuid", "source", "peak_frequency_hz", "audio_path"] if c in df.columns]
    display(df[cols].head(N) if cols else df.head(N))